# Day 5: Agentic Workflows

**LLMs for Social Science** | Oxford Spring School

## The Course at a Glance

| Day | Theme | What You Built |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | BoW → GloVe embeddings → tokenization → attention → generation |
| Day 2 ✓ | From Models to Tools | Post-training, prompting as experimental design, reasoning |
| Day 3 ✓ | Deploying for Research | Validation (Cohen's κ), fine-tuning, APIs at scale |
| Day 4 ✓ | Social Science Applications | Information extraction, RAG pipeline, simulated agents |
| **→ Day 5** | **Agentic Workflows** | **Tool use, autonomous research assistants, production tools** |

Yesterday you built a **RAG pipeline**: chunk documents → embed → index in FAISS → retrieve → generate an answer grounded in your corpus. You also extracted structured information from academic text and verified faithfulness.

Today those functions become **tools** that a model can call autonomously. Instead of you deciding "first retrieve, then extract, then compare," the model makes that plan itself. This is the core idea behind agents.

**Why this matters for research:**

- Literature review requires multi-step reasoning: find papers → extract claims → compare across sources → synthesize. A single prompt can't do this reliably. An agent loop can.
- Understanding the agent architecture helps you evaluate tools like Claude Code and Elicit — they use exactly the pattern you'll build today.
- The gap between "cool demo" and "reliable research tool" is entirely about failure modes and safeguards. You'll learn both.

## Today's outline

| Section | Topic | Time |
|---------|-------|------|
| **Setup** | Reconnect to the Day 4 corpus and API | ~10 min |
| **Section 1** | What Is an Agent? Tool definitions + manual planning | ~35 min |
| **Section 2** | Build a ReAct Agent — the loop, running it, stress-testing | ~60 min |
| *Break* | | ~15 min |
| **Section 3** | From Notebooks to Production (live demo) | ~25 min |
| **Closing** | The full arc + where to go from here | ~15 min |


## Setup

Today is **API-only** where possible: we use a large model via the Nebius API for everything. No GPU memory management. This is deliberate — agent loops require a model that reliably follows structured formats. Smaller models (3B–7B) struggle with ReAct's Thought/Action pattern and produce format errors that obscure the lesson.

**Preferred:** Nebius Token Factory (same key as Day 4) — uses Qwen3-235B, excellent quality  
**Fallback:** If no Nebius key is found, the notebook automatically loads a small local model (Qwen2.5-3B, 4-bit quantised) using `transformers`. Quality will be lower but the loop still runs.

Add your Nebius key to Colab secrets (key icon in the left sidebar) as `oss26_key`.

The setup reuses your Day 4 infrastructure: the same academic corpus, the same embedding model, the same FAISS index.


In [1]:
#@title Install dependencies
!pip install -q openai sentence-transformers faiss-cpu
!pip install -q pandas numpy tqdm PyMuPDF
!pip install -q transformers accelerate bitsandbytes  # for local fallback

import os
import json
import re
import textwrap
import pandas as pd
import numpy as np
from tqdm import tqdm

print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
Dependencies installed.


In [2]:
#@title API setup — Nebius (cloud) → local transformers (fallback)

# ── Configuration ────────────────────────────────────────────────────────────
NEBIUS_MODEL      = "deepseek-chat"
LOCAL_MODEL_NAME  = "Qwen/Qwen2.5-3B-Instruct"   # ← change to swap local model
# ─────────────────────────────────────────────────────────────────────────────

from google.colab import userdata

PROVIDER = None

# ── 1. Try Nebius ─────────────────────────────────────────────────────────────
try:
    NEBIUS_API_KEY = userdata.get("API_KEY")
    if NEBIUS_API_KEY:
        from openai import OpenAI
        api_client = OpenAI(
            base_url="https://api.deepseek.com/v1/",
            api_key=NEBIUS_API_KEY,
        )
        PROVIDER = "nebius"
        print(f"✓ Using Nebius API (model: {NEBIUS_MODEL})")
except Exception as e:
    print(f"Nebius unavailable: {e}")

# ── 2. Fall back to local transformers ────────────────────────────────────────
if PROVIDER is None:
    print(f"Loading local model: {LOCAL_MODEL_NAME} (4-bit quant) ...")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    _bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    _tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    _model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME,
        quantization_config=_bnb_config,
        device_map="auto",
    )
    PROVIDER = "local"
    print(f"✓ Loaded {LOCAL_MODEL_NAME} on {_model.device}")


# ── Unified generate interface ────────────────────────────────────────────────

def generate_chat(
    messages: list[dict],
    max_tokens: int = 1024,
    temperature: float = 0.0,
) -> str:
    """Generate from a full conversation history (list of role/content dicts)."""
    if PROVIDER == "nebius":
        response = api_client.chat.completions.create(
            model=NEBIUS_MODEL,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content.strip()

    elif PROVIDER == "local":
        do_sample = temperature > 0
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        with torch.no_grad():
            output = _model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature if do_sample else 1.0,
                do_sample=do_sample,
                pad_token_id=_tokenizer.eos_token_id,
            )
        return _tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()


def generate(
    prompt: str,
    system: str = "You are a helpful research assistant.",
    max_tokens: int = 1024,
    temperature: float = 0.0,
) -> str:
    """Convenience wrapper for single-turn prompts."""
    return generate_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )


# ── Smoke test ────────────────────────────────────────────────────────────────
print(generate("Say 'API working' and nothing else."))


✓ Using Nebius API (model: deepseek-chat)
API working


In [ ]:
#@title Download and extract the academic corpus (same as Day 4)
# Annual Review of Political Science, 2021–2024
# ~100 articles as PDFs, extracted with PyMuPDF, chunked, embedded, indexed in FAISS

import fitz  # PyMuPDF
import zipfile
import urllib.request
from sentence_transformers import SentenceTransformer
import faiss

# --- Download PDFs ---
BASE_URL = "https://github.com/antndlcrx/Intro-to-LLMs-DPIR/raw/main/data/articles"
YEARS = [2021, 2022, 2023, 2024]
ARTICLES_DIR = "articles"
os.makedirs(ARTICLES_DIR, exist_ok=True)

for year in YEARS:
    pdf_dir = os.path.join(ARTICLES_DIR, str(year))
    if not os.path.exists(pdf_dir):
        url = f"{BASE_URL}/{year}.zip"
        zip_path = os.path.join(ARTICLES_DIR, f"{year}.zip")
        print(f"Downloading {year}.zip...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(ARTICLES_DIR)
        print(f"  → {len(os.listdir(pdf_dir))} PDFs extracted")
    else:
        print(f"{year}: already downloaded ({len(os.listdir(pdf_dir))} PDFs)")


# --- Extract text from PDFs ---
def extract_articles(articles_dir, years):
    """Extract text from all PDFs. Returns dict: article_id -> {title, text, year, ...}"""
    articles = {}
    for year in years:
        pdf_dir = os.path.join(articles_dir, str(year))
        if not os.path.exists(pdf_dir):
            continue
        for filename in sorted(os.listdir(pdf_dir)):
            if not filename.endswith('.pdf'):
                continue
            doc = fitz.open(os.path.join(pdf_dir, filename))
            pages_text = []
            for page in doc:
                text = page.get_text()
                lines = [l for l in text.split('\n')
                         if not l.startswith('Downloaded from www.annualreviews.org')]
                pages_text.append('\n'.join(lines))
            full_text = '\n'.join(pages_text)

            # Parse title from first page
            first_page_lines = [l.strip() for l in pages_text[0].split('\n') if l.strip()]
            title_lines = []
            capture = False
            for line in first_page_lines:
                if 'Annual Review of Political Science' in line:
                    capture = True
                    continue
                if capture:
                    if any(x in line for x in ['email:', 'Department of', 'Annu. Rev.',
                                                'First published', 'Copyright', 'licensed']):
                        if title_lines:
                            break
                        continue
                    if line and len(line) > 2:
                        title_lines.append(line)
                        if len(title_lines) >= 3:
                            break

            article_id = filename.replace('.pdf', '')
            articles[article_id] = {
                'title': ' '.join(title_lines).strip(),
                'filename': filename,
                'text': full_text,
                'pages': len(doc),
                'year': year,
            }
            doc.close()
    return articles


articles = extract_articles(ARTICLES_DIR, YEARS)
print(f"\nLoaded {len(articles)} articles across {len(YEARS)} years")
for year in YEARS:
    year_articles = [a for a in articles.values() if a['year'] == year]
    print(f"  {year}: {len(year_articles)} articles")

In [ ]:
#@title chunk and embed


# --- Chunk ---
def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping chunks of ~chunk_size words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if len(chunk.split()) > 30:
            chunks.append(chunk)
        start = end - overlap
    return chunks

all_chunks = []
for article_id, article in articles.items():
    for i, chunk in enumerate(chunk_text(article['text'])):
        all_chunks.append({
            "text": chunk,
            "source": article_id,
            "title": article['title'],
            "year": article['year'],
            "chunk_id": i,
        })

corpus_df = pd.DataFrame(all_chunks)
print(f"\nCorpus: {len(corpus_df)} chunks from {len(articles)} articles")
print(f"Average chunk: {corpus_df['text'].str.split().str.len().mean():.0f} words")


# --- Embed and index ---
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
chunk_embeddings = embed_model.encode(
    corpus_df['text'].tolist(), show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(chunk_embeddings)

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print(f"\nFAISS index ready: {index.ntotal} vectors, dim={chunk_embeddings.shape[1]}")


In [ ]:
#@title Core functions from Day 4

def retrieve(query, k=5):
    """
    Retrieve the top-k most relevant chunks for a query.
    Uses the FAISS index built above. Returns a list of dicts.
    """
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = corpus_df.iloc[idx]
        results.append({
            'text': row['text'],
            'source': row['source'],
            'title': row.get('title', ''),
            'year': row.get('year', ''),
            'chunk_id': row['chunk_id'],
            'score': float(score),
        })
    return results

def rag_answer(question, k=5):
    """Retrieve relevant passages, then generate an answer grounded in them."""
    results = retrieve(question, k=k)
    context = "\n\n".join(
        f"[Source: {r['source']}] {r['text']}" for r in results
    )
    prompt = (
        f"Based on the following passages from political science articles, "
        f"answer this question: {question}\n\n"
        f"Passages:\n{context}\n\n"
        f"Answer based only on the passages above. If the passages don't contain "
        f"enough information, say so."
    )
    return generate(prompt, max_tokens=600)

# Quick test
test_results = retrieve("democratic backsliding", k=2)
print("Retrieval test:")
for r in test_results:
    print(f"  [{r['source']}] (score={r['score']:.3f}) {r['text'][:100]}...")


---

# Section 1: What Is an Agent?

In every session so far, **you** have been the decision-maker. You decided which prompt to write, which tool to call, which results to inspect. The model did what you asked, one step at a time.

An **agent** reverses this. You give the model a goal, and it decides what steps to take: which tools to call, in what order, and what to do with the results. The model becomes the planner; you become the supervisor.

Most agents follow a loop called **ReAct** (Reasoning + Acting):

```
1. THOUGHT:      The model reasons about what to do next
2. ACTION:       The model calls a tool (search, extract, compare, etc.)
3. OBSERVATION:  The tool returns a result
4. REPEAT        until the model has enough information
5. FINAL ANSWER: The model synthesizes a response
```

This is not magic. It is a **prompt format** combined with a **while loop**. The model generates structured text, your code parses it and executes the requested tool, and you feed the result back. That's it.

Before we automate this, you do it by hand.


## The agent's toolkit

An agent is only as useful as its tools. For Day 5, the tools come directly from your Day 4 work:

| Tool | What it does | Day 4 origin |
|------|-------------|--------------|
| `search_literature` | Semantic search over ~100 political science articles | `retrieve()` + FAISS |
| `extract_findings` | Pull structured claims from a text passage | Extraction prompting |
| `summarize` | Condense a passage into key points | Summarization prompting |
| `calculate` | Evaluate a mathematical expression | New (simple) |

Let's define them.


In [ ]:
#@title Define the agent's tools

# --- Tool 1: Search the literature ---
def search_literature(query):
    """Search the political science corpus. Returns the 3 most relevant passages."""
    results = retrieve(query, k=3)
    if not results:
        return "No relevant passages found."
    output = []
    for i, r in enumerate(results, 1):
        output.append(f"[{i}] Source: {r['source']} ({r.get('year', '?')}) — {r.get('title', 'Untitled')[:80]}\n"
                       f"    Relevance: {r['score']:.2f}\n{r['text'][:500]}")
    return "\n\n".join(output)


# --- Tool 2: Extract structured findings ---
def extract_findings(text):
    """Extract the main empirical finding or theoretical claim from a passage."""
    prompt = (
        "Extract the main finding or claim from this political science text. "
        "Return a JSON object with keys: 'claim' (one sentence), "
        "'evidence_type' (empirical/theoretical/review), "
        "'topic' (2-3 word topic label).\n\n"
        f"Text: {text[:1000]}\n\nJSON:"
    )
    return generate(prompt, max_tokens=200)


# --- Tool 3: Summarize ---
def summarize(text):
    """Summarize a passage in 2-3 sentences, focusing on the main argument."""
    prompt = (
        "Summarize the following passage from a political science article "
        "in 2-3 sentences. Focus on the main argument or finding.\n\n"
        f"Text: {text[:1500]}\n\nSummary:"
    )
    return generate(prompt, max_tokens=150)


# --- Tool 4: Calculator ---
def calculate(expression):
    """Evaluate a mathematical expression. Input: a valid Python expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# --- Tool registry ---
# The agent reads this to know what tools are available and how to call them.
TOOLS = {
    "search_literature": {
        "function": search_literature,
        "description": (
            "Search the political science article corpus for passages relevant to a query. "
            "Input: a natural language research question or topic (e.g., 'democratic backsliding in Europe'). "
            "Returns the 3 most relevant passages with source information."
        ),
    },
    "extract_findings": {
        "function": extract_findings,
        "description": (
            "Extract the main finding or claim from a text passage as structured JSON. "
            "Input: the text passage to analyze."
        ),
    },
    "summarize": {
        "function": summarize,
        "description": (
            "Summarize a text passage in 2-3 sentences, focusing on the main argument. "
            "Input: the text passage to summarize."
        ),
    },
    "calculate": {
        "function": calculate,
        "description": (
            "Evaluate a mathematical expression. "
            "Input: a valid Python math expression (e.g., '0.73 * 1000')."
        ),
    },
}

print("Agent toolkit:")
for name, info in TOOLS.items():
    print(f"  {name}: {info['description'][:80]}...")


### Exercise 1a: Think like an agent

Before building the agent loop, practice the reasoning yourself. This is "Be the Algorithm" for agents: you plan the steps a model would take.

Given the research question below and the four tools above, write out the steps you would take. Use the Thought / Action / Observation format. You don't run anything yet — just plan.

**Research question:** *"What does recent political science research say about the relationship between social media and political polarization?"*


In [ ]:
### Exercise 1a: Plan a multi-step research task

# Research question: "What does recent political science research say about
# the relationship between social media and political polarization?"
#
# Write your plan using the Thought / Action / Observation format.
# You DON'T run the tools yet — just think through the steps.
#
# Thought 1: [what do you need to find out first?]
# Action 1:  [which tool? what input?]
# Observation 1: [what do you expect to get back?]
#
# Thought 2: [given that observation, what next?]
# Action 2:  [which tool? what input?]
# Observation 2: [what do you expect?]
#
# ... continue until you have enough to answer ...
#
# Final Answer: [how would you synthesize the answer?]

# YOUR PLAN HERE:
#
#
#


In [ ]:
#@title Solution: Exercise 1a

print("""
A reasonable plan:

Thought 1: I need to find what the corpus says about social media and polarization.
Action 1:  search_literature("social media political polarization")
Observation 1: [Several passages about social media effects on polarization]

Thought 2: Let me also search for the specific mechanism — echo chambers or
           algorithmic amplification — to get more targeted results.
Action 2:  search_literature("echo chambers filter bubbles polarization")
Observation 2: [Passages about echo chamber theory and evidence]

Thought 3: I have passages from multiple sources. Let me extract the main
           finding from the most relevant one to get structured information.
Action 3:  extract_findings([text of the most relevant passage])
Observation 3: [Structured JSON with claim, evidence type, topic]

Thought 4: I now have enough information from multiple sources to synthesize.
Final Answer: [Answer citing specific articles and findings from the corpus]

Key observations:
  - The plan requires MULTIPLE searches with different queries, not just one.
    A single search might miss relevant work that uses different terminology.
  - extract_findings gives structure, but the agent must decide WHICH passage
    to extract from — it uses judgment, not just calling every tool on everything.
  - The final answer should cite sources from the observations, not hallucinate.
""")


### Exercise 1b: Test the tools manually

Before the agent calls these tools, verify they work. Run each tool on an input you choose. This is the same principle as Day 3: validate your instrument before trusting its output.


In [ ]:
### Exercise 1b: Test each tool manually

# 1. Search the literature — try a research question
results = search_literature("democratic backsliding")  # <-- replace with your query
print("=== Search Results ===")
print(results[:500])

# Hints:
# - Try: "democratic backsliding", "voter turnout inequality",
#   "populism and democratic institutions"
# - Are the results relevant? Would they help answer a research question?


In [ ]:
results

In [ ]:
# 2. Extract findings from one of the passages you just retrieved
# Copy a passage from the search results above and paste it here

passage = "[1] Source: annurev-polisci-041322-025352 (2024) — Theories of Democratic Backsliding Edoardo Grillo,1 Zhaotian Luo,2 Monika Nalepa\n    Relevance: 0.69\ncan decrease the risk of backsliding via aggrandizement (restrainers are more likely to oppose aggrandizement in Howell et al. 2023), but not under electoral manipulation (parties are less likely to sanction their own charismatic leaders in Hollyer et al. 2011). A third implication concerns the electorate’s normative attachments to democratic values.Con- trary to what conventional wisdom might suggest, strengthening voters’ democratic values can increase the risk of backsliding, as shown, via di\n\n"  # <-- replace
findings = extract_findings(passage)
print("=== Extracted Findings ===")
print(findings)

# Does the extraction look right? Is the claim faithful to the passage?


In [ ]:
# 3. Try summarize and calculate

summary = summarize("Source: annurev-polisci-041322-025352 (2024) — Theories of Democratic Backsliding Edoardo Grillo,1 Zhaotian Luo,2 Monika Nalepa\n    Relevance: 0.63\nhighlights that the gradual dismantling of democracy is more than just the flip side of democratic consolidation. The goal of this review is to take stock of this body of work and improve its accessibility to researchers, particularly those with more empirical agendas. Specifically, the review clarifies how formal models explain democratic backsliding and how they answer questions such as: Does polit- ical polarization contribute to backsliding? Are more popular and electorally secure incumbents\n\n")  # <-- replace
print("=== Summary ===")
print(summary)

print("\n=== Calculator ===")
print(calculate("0.73 * 1500"))  # sample size * effect
print(calculate("1 / 0.05"))     # inverse of p-value threshold


---

# Section 2: Build a ReAct Agent

You've planned tool calls by hand and verified the tools work. Now you automate it. The agent has three components:

1. **A system prompt** that tells the model about its tools and the Thought/Action format
2. **A parser** that extracts the tool name and input from the model's text output
3. **A loop** that executes tools, feeds results back, and repeats until done

You write the system prompt (the hardest part). The parser and executor are provided — they're string parsing, not conceptual. The loop is the exercise.


### Exercise 2a: Write the agent system prompt

This is the most important piece. The system prompt must:

1. Explain the agent's role
2. List the available tools with their descriptions
3. Specify the **exact format** for Thought / Action / Action Input
4. Explain when to give a Final Answer
5. Set rules: one tool at a time, don't invent information, cite sources

The format must be **precise** — the parser will look for exact strings like `Action:` and `Action Input:`. If the model deviates, the loop breaks.


In [ ]:
### Exercise 2a: Write the agent system prompt

# Build tool descriptions from the registry (this is provided)
tool_descriptions = ""
for name, info in TOOLS.items():
    tool_descriptions += f"  - {name}: {info['description']}\n"

# YOUR CODE: Write the system prompt
# It should include all 5 elements listed above.
# Be explicit about the format — the model follows it literally.

AGENT_SYSTEM_PROMPT = ""  # <-- YOUR PROMPT HERE

print(AGENT_SYSTEM_PROMPT)


In [ ]:
#@title Solution: Exercise 2a

tool_descriptions = ""
for name, info in TOOLS.items():
    tool_descriptions += f"  - {name}: {info['description']}\n"

AGENT_SYSTEM_PROMPT = f"""You are a political science research assistant. You answer research questions by searching a corpus of political science articles and analyzing the results.

You have access to these tools:
{tool_descriptions}
To use a tool, respond in this EXACT format:

Thought: [your reasoning about what to do next]
Action: [tool_name]
Action Input: [the input to the tool]

You will then receive an Observation with the tool's result. Continue with another Thought.

When you have enough information to fully answer the question, respond with:

Thought: I now have enough information to answer the question.
Final Answer: [your complete answer, citing the sources you found]

Rules:
- Always start with a Thought before any Action.
- Use exactly ONE tool per step.
- Do NOT make up information. Use only what the tools return.
- If the tools do not return relevant information, say so honestly.
- In your Final Answer, cite the article sources from the search results.
"""

print(AGENT_SYSTEM_PROMPT)


### Parser and executor (provided)

These are plumbing — string parsing and function dispatch. Read through them to understand what they do, but don't write them from scratch. The interesting part is the loop.


In [ ]:
#@title Parser: extract Action and Action Input from model output

def parse_agent_response(response):
    """
    Parse the model's response to extract either a tool call or a final answer.

    Returns a dict:
      {"type": "action", "tool": name, "input": input_text}
      {"type": "final_answer", "answer": answer_text}
      {"type": "error", "message": description}
    """
    # Check for Final Answer first
    if "Final Answer:" in response:
        answer = response.split("Final Answer:")[-1].strip()
        return {"type": "final_answer", "answer": answer}

    # Look for Action: and Action Input:
    action_match = re.search(r'Action:\s*(.+)', response)
    input_match = re.search(r'Action Input:\s*(.+)', response, re.DOTALL)

    if action_match and input_match:
        tool_name = action_match.group(1).strip()
        # Action Input may be multi-line (e.g., a long passage)
        tool_input = input_match.group(1).strip()
        # Clean up: take only until the next "Thought:" or "Action:" if present
        for stop_token in ["\nThought:", "\nAction:", "\nObservation:"]:
            if stop_token in tool_input:
                tool_input = tool_input[:tool_input.index(stop_token)].strip()
        return {"type": "action", "tool": tool_name, "input": tool_input}

    return {"type": "error", "message": f"Could not parse response: {response[:200]}"}


# Quick test
test_cases = [
    "Thought: I need to search.\nAction: search_literature\nAction Input: democratic backsliding",
    "Thought: Done.\nFinal Answer: The evidence suggests polarization has increased.",
    "This is not a valid agent response.",
]
for tc in test_cases:
    print(f"  {parse_agent_response(tc)['type']}: {str(parse_agent_response(tc))[:80]}")


In [ ]:
#@title Executor: dispatch tool calls

def execute_tool(tool_name, tool_input):
    """Call a tool by name. Returns the tool's output as a string."""
    if tool_name not in TOOLS:
        return f"Error: Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"

    try:
        result = TOOLS[tool_name]["function"](tool_input)
        # Truncate very long results to avoid context overflow
        if len(str(result)) > 2000:
            result = str(result)[:2000] + "\n... [truncated]"
        return str(result)
    except Exception as e:
        return f"Error calling {tool_name}: {e}"


In [ ]:
search_literature()

In [ ]:
execute_tool("search_literature", "democratic backsliding")

### Exercise 2b: The agent loop

Now build the loop. The structure is:

```python
while steps < max_steps:
    response = generate(...)         # model produces Thought + Action
    parsed = parse(response)         # extract tool name and input
    if parsed is final_answer: return
    if parsed is action: execute tool, feed result back
    if parsed is error: ask model to try again
```

You write the loop. The helper functions (`generate`, `parse_agent_response`, `execute_tool`) are already defined.


In [ ]:
### Exercise 2b: The agent loop

def run_agent(question, max_steps=6, verbose=True):
    """
    Run the ReAct agent on a research question.

    Args:
        question: The research question.
        max_steps: Maximum tool calls before stopping.
        verbose: Print each step.

    Returns:
        The agent's final answer, or a timeout message.
    """
    # YOUR CODE HERE
    # 1. Initialize the conversation (system prompt + user question)
    # 2. Loop up to max_steps times:
    #    a. Call generate() with the full conversation
    #    b. Parse the response
    #    c. If final_answer → return it
    #    d. If action → execute the tool, append the exchange to conversation
    #    e. If error → append a recovery message
    # 3. If loop ends without final answer → return a timeout message

    pass  # <-- replace with your implementation


# Hints:
# - The conversation is a list of messages, just like Day 2:
#   [{"role": "system", "content": AGENT_SYSTEM_PROMPT},
#    {"role": "user", "content": question}]
# - After the model takes an action and you get an observation,
#   append the model's response as {"role": "assistant", "content": response}
#   and the observation as {"role": "user", "content": f"Observation: {observation}"}
# - Use generate_chat(messages, max_tokens=600) to get the model's response.
#   It takes the full messages list and returns a string.


In [ ]:
#@title Solution: Exercise 2b

def run_agent(question, max_steps=6, verbose=True):
    """Run the ReAct agent on a research question."""

    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(max_steps):
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"STEP {step + 1}")
            print(f"{'=' * 60}")

        # Generate the model's next move
        response = generate_chat(messages, max_tokens=600, temperature=0.0)

        if verbose:
            print(f"MODEL:\n{response[:600]}")

        # Parse it
        parsed = parse_agent_response(response)

        if parsed["type"] == "final_answer":
            if verbose:
                print(f"\n{'=' * 60}")
                print(f"FINAL ANSWER:\n{parsed['answer']}")
            return parsed["answer"]

        elif parsed["type"] == "action":
            tool_name = parsed["tool"]
            tool_input = parsed["input"]

            if verbose:
                print(f"\n→ Calling: {tool_name}({tool_input[:80]}{'...' if len(tool_input) > 80 else ''})")

            observation = execute_tool(tool_name, tool_input)

            if verbose:
                print(f"← Result: {observation[:300]}{'...' if len(observation) > 300 else ''}")

            # Append exchange to conversation history
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": f"Observation: {observation}"})

        elif parsed["type"] == "error":
            if verbose:
                print(f"⚠ Parse error: {parsed['message']}")
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user", "content": (
                "Your response was not in the correct format. "
                "Please respond with either:\n"
                "Thought: ...\nAction: ...\nAction Input: ...\n"
                "OR:\nThought: ...\nFinal Answer: ..."
            )})

    return "Agent reached maximum steps without producing a final answer."


### Exercise 2c: Run the agent

Let the agent loose on a research question. Watch it reason, act, and synthesize. Then try your own questions.


In [ ]:
### Exercise 2c: Run the agent

# Start with this question, then try your own
question = "What does recent political science research say about the relationship between social media and political polarization?"

answer = run_agent(question, max_steps=6, verbose=True)


In [ ]:
# Try more questions — these require different tool combinations

# A question that needs multiple searches
answer = run_agent(
    "Compare what the literature says about the causes of populism in "
    "Europe vs. Latin America.",
    max_steps=6, verbose=True
)

# A question the corpus probably can't answer
# answer = run_agent(
#     "What is the current GDP of France?",
#     max_steps=4, verbose=True
# )

# A question that might benefit from extract_findings
# answer = run_agent(
#     "What are the main empirical findings on voter turnout and inequality?",
#     max_steps=6, verbose=True
# )


In [ ]:
#@title Solution: Exercise 2c — what to look for

print("""
Evaluating the agent's behavior:

GOOD SIGNS:
  ✓ The agent searches first, then synthesizes (not the reverse)
  ✓ It uses multiple searches with different query phrasings
  ✓ The Final Answer cites specific sources from the Observations
  ✓ It uses extract_findings or summarize when passages are long
  ✓ When documents lack the answer, it says so

PROBLEMS TO WATCH FOR:
  ✗ Jumps to Final Answer without searching (skips tools entirely)
  ✗ Repeats the same search query multiple times
  ✗ Final Answer includes claims not found in any Observation
  ✗ Gets stuck in a loop (max_steps saves you here)
  ✗ Calls a tool with the wrong kind of input

Compare: how does the agent's answer compare to a single-shot RAG call?
""")

# Direct comparison: agent vs. single-shot RAG
print("\n" + "=" * 60)
print("SINGLE-SHOT RAG (no agent loop):")
print("=" * 60)
direct_answer = rag_answer(
    "What does recent political science research say about the "
    "relationship between social media and political polarization?"
)
print(direct_answer)
print("\n→ The agent can search multiple times and synthesize across searches.")
print("  Single-shot RAG is limited to whatever the first retrieval returns.")


---

*Take a 15-minute break. When we come back: stress-testing and failure modes.*

---


### Exercise 2d: Stress-test the agent

Agents fail in ways that are harder to detect than prompting failures. The model can confabulate tool results, loop indefinitely, or drift out of the ReAct format. You need to know what breaks and how to catch it.

Run these stress tests and observe what happens.


In [ ]:
### Exercise 2d: Stress-test the agent

# Test 1: A question the corpus cannot answer
print("TEST 1: Out-of-domain question")
print("-" * 50)
answer1 = run_agent(
    "What was the unemployment rate in Spain in March 2024?",
    max_steps=4, verbose=True
)
print(f"\nAnswer: {answer1[:200]}")

print("\n\n")

# Test 2: A very broad question that could use many tool calls
print("TEST 2: Very broad question")
print("-" * 50)
answer2 = run_agent(
    "Summarize all major findings about democracy from the past 4 years "
    "of political science research.",
    max_steps=6, verbose=True
)
print(f"\nAnswer: {answer2[:300]}")


In [ ]:
# Test 3: YOUR stress test
# Design a question that you think will cause the agent to fail.
# Think about: ambiguous queries, adversarial inputs, questions that
# require tools the agent doesn't have, very long multi-step reasoning.

# YOUR QUESTION HERE:
# answer3 = run_agent("...", max_steps=4, verbose=True)


In [ ]:
#@title Solution: Exercise 2d — failure mode taxonomy

print("""
Common agent failure modes (what you should have observed):

1. HALLUCINATION ON MISSING DATA
   When the corpus doesn't contain the answer, the agent should say
   "I could not find this information." A good agent does this; a bad
   one invents an answer from training data.
   → Safeguard: explicit system prompt rule + verify Final Answer
     cites Observations.

2. OVER-SEARCHING
   For broad questions, the agent may search 4-5 times without
   converging. Each search adds to the context window, and eventually
   the model loses track of early results.
   → Safeguard: max_steps limit. In production, also summarize
     intermediate results every N steps.

3. TOOL MISUSE
   The agent calls a tool with invalid input (e.g., passing a question
   to calculate, or a math expression to search_literature).
   → Safeguard: input validation in execute_tool + error recovery
     prompt in the loop.

4. CONTEXT OVERFLOW
   After many steps, the full conversation exceeds the model's
   effective attention span. Early observations get "forgotten."
   → Safeguard: truncate old messages, or periodically summarize
     the conversation so far.

5. PREMATURE TERMINATION
   The agent gives a Final Answer after just one search, even when
   the question requires multiple perspectives.
   → Safeguard: you can add a "minimum steps" or explicitly prompt
     "search for at least two different perspectives before answering."
""")


### Exercise 2e: Improve the agent (stretch)

Pick **one** improvement and implement it. Modify `run_agent` or the system prompt accordingly.

Options:
- **Minimum evidence rule**: require the agent to search at least twice before giving a Final Answer. Add a check in the loop.
- **Step budget awareness**: after step 3, inject a message: "You have used 3 of 6 steps. Summarize what you've found so far and plan your remaining steps."
- **Source verification**: after the agent gives a Final Answer, automatically check whether every claim in the answer appears in the Observations. (Hint: you can call `generate()` to do this check.)
- **Better error recovery**: when the parser fails, instead of a generic "wrong format" message, show the model its own output and explain what went wrong.


In [ ]:
### Exercise 2e: Choose one improvement and implement it

# YOUR CODE HERE

# Example skeleton for "step budget awareness":
#
# def run_agent_v2(question, max_steps=6, verbose=True):
#     messages = [...]
#     for step in range(max_steps):
#         ...
#         # After step 3, add a budget message
#         if step == 2:
#             messages.append({"role": "user", "content":
#                 f"You have used {step+1} of {max_steps} steps. "
#                 "Briefly summarize what you've found so far, then "
#                 "plan how to use your remaining steps efficiently."
#             })
#         ...


**Section takeaway:** An agent is a loop, not magic. The quality depends on three things: (1) the model's ability to follow the Thought/Action format reliably, (2) the quality and coverage of the tools, and (3) safeguards against the failure modes you just catalogued. In production, you add retry logic, context management, cost tracking, and human-in-the-loop approval for high-stakes actions.

---


# Section 3: From Notebooks to Production

## The gap

The agent you built works, but it lives in a Colab notebook. Production agents need:

- **A real development environment** — a terminal, file system, version control (not a notebook)
- **Persistent tool connections** — databases, APIs, web search, file I/O (not Python functions)
- **Better observability** — logging every step, tracking costs, auditing tool calls
- **Human oversight** — approval gates for high-stakes actions (sending emails, modifying data)

Three tools bridge this gap. Your instructor will demo each one live.


## GitHub Codespaces

[Codespaces](https://github.com/features/codespaces) gives you a full cloud development environment in your browser: VS Code with a terminal, file system, Git, Python, and anything else you need. No local installation required.

For agent development, Codespaces solves the "works on my machine" problem. You configure the environment once (in a `devcontainer.json` file), and every collaborator gets an identical setup.

**To try it yourself:**
1. Go to [github.com/codespaces](https://github.com/codespaces)
2. Create a new Codespace from a blank template or your own repo
3. You get a full Linux terminal in the browser

GitHub's free tier includes 60 hours/month of Codespace usage.


## Claude Code

[Claude Code](https://docs.anthropic.com/en/docs/claude-code) is a command-line agent that lives in your terminal. It can read and write files, execute shell commands, search codebases, and plan multi-step tasks.

For researchers, Claude Code is useful for:
- **Data analysis**: "Load this CSV, compute summary statistics, and create a visualization"
- **Pipeline building**: "Write a Python script that cleans and merges these three datasets"
- **Iterative analysis**: "The regression shows heteroskedasticity. Add robust standard errors and re-run"

It uses the exact same architecture you just built — ReAct with tool use — but with a frontier model and real filesystem tools.

**Installation** (in a Codespace or local terminal):
```
npm install -g @anthropic-ai/claude-code
```

**Usage:**
```
cd my-research-project/
claude
```
Then talk to it: "Read the data in results.csv and plot the distribution of the stance variable."


## Model Context Protocol (MCP)

[MCP](https://modelcontextprotocol.io) is an open standard that lets AI models connect to external tools and data sources. Instead of defining tools as Python functions (as you did today), MCP defines a protocol so that any tool can work with any model.

Think of it as **USB for AI**: just as USB lets any peripheral work with any computer, MCP lets any tool work with any AI model.

**Why it matters for research:**
- Connect Claude to your university's database, Google Drive, or Zotero
- Build custom MCP servers that expose your datasets or analysis pipelines
- Share tools across your research group: one person builds the server, everyone uses it

**Architecture:**
```
Your AI model  ←—MCP—→  MCP Server  ←→  Your data source
                        (translates between
                         MCP protocol and your data)
```

MCP servers are lightweight programs (often 50–100 lines of Python or TypeScript). The [MCP documentation](https://modelcontextprotocol.io) has quickstart guides and examples.


### Live demo notes

*Your instructor will demonstrate:*

1. **GitHub Codespaces** — opening a cloud environment, navigating the terminal, running Python
2. **Claude Code** — installing in the Codespace, giving it a research task, watching it plan and execute
3. **MCP** (brief) — how Claude Code connects to external data through MCP servers

*After the demo:*
- Codespaces: [github.com/codespaces](https://github.com/codespaces) (free tier: 60h/month)
- Claude Code: needs an [Anthropic API key](https://console.anthropic.com/)
- MCP: [modelcontextprotocol.io](https://modelcontextprotocol.io)


---

# Closing: The Full Arc

## Five days, one toolkit

| Day | What You Built | Key Insight |
|-----|----------------|-------------|
| **Day 1** | BoW → embeddings → attention → generation | LLMs are next-token predictors. Everything else builds on this. |
| **Day 2** | Post-training, prompting, reasoning | Prompting is experimental design. Small wording changes → big output changes. |
| **Day 3** | Validation (κ), fine-tuning (DeBERTa, LoRA) | Never trust model output without validation. Fine-tune when prompting hits a ceiling. |
| **Day 4** | Information extraction, RAG pipeline, simulation | RAG grounds models in your data. Simulation requires careful validity assessment. |
| **Day 5** | ReAct agent with tool use | An agent is a loop, not magic. Quality = model capability × tool quality × safeguards. |

The progression has been deliberate: from understanding how models work (Day 1), to controlling them (Days 2–3), to applying them to research tasks (Day 4), to orchestrating them autonomously (Day 5). Each day's tools become the next day's building blocks.


## Where to go from here

**Immediate next steps:**
- Apply the RAG pipeline (Day 4) to your own corpus — parliamentary records, news archives, legal texts
- Build a classification pipeline (Day 3) for your own annotation task
- Try Claude Code on a real research project
- Set up a Codespace for your research group

**Resources:**
- [Anthropic API documentation](https://docs.anthropic.com) — Claude models
- [OpenAI API documentation](https://platform.openai.com/docs) — GPT models
- [Hugging Face](https://huggingface.co) — open models, datasets, tutorials
- [MCP documentation](https://modelcontextprotocol.io) — building tool connections
- [Claude Code docs](https://docs.anthropic.com/en/docs/claude-code) — terminal-based agents
- [LangGraph](https://langchain-ai.github.io/langgraph/) — complex agent architectures

**Stay connected:**
- Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)
- All notebooks and materials remain available after the course
- If you're interested in doing research on LLMs for social science, reach out — we're building a research lab focused on these methods

Thank you for five days of hard work. You now have the foundations to use language models rigorously in your research.
